In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import re
from collections import Counter
from typing import Tuple

In [7]:
data = pd.read_csv("ml_challenge_dataset.csv")

print("first 4 rows of data:")
data.head()

first 4 rows of data:


,unique_id,Painting,"On a scale of 1–10, how intense is the emotion conveyed by the artwork?",Describe how this painting makes you feel.,This art piece makes me feel sombre.,This art piece makes me feel content.,This art piece makes me feel calm.,This art piece makes me feel uneasy.,How many prominent colours do you notice in this painting?,How many objects caught your eye in the painting?,How much (in Canadian dollars) would you be willing to pay for this painting?,"If you could purchase this painting, which room would you put that painting in?","If you could view this art in person, who would you want to view it with?",What season does this art piece remind you of?,"If this painting was a food, what would be?",Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.
0,1,The Persistence of Memory,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,The Persistence of Memory,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,4.0,0,Bathroom,By yourself,Fall,Fries,A country song that contrasts nostalgia for th...
2,3,The Persistence of Memory,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,3.0,$5,"Bathroom,Dining room","Coworkers/Classmates,By yourself",Fall,Sardines,A melancholy instrumental with a monotone voic...
3,4,The Persistence of Memory,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,7.0,a,"Bedroom,Bathroom",Coworkers/Classmates,Winter,a,q
4,5,The Persistence of Memory,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,6.0,300 dollars.,Living room,Friends,"Spring,Summer",Churros.,Radiohead's album in rainbows.


In [8]:
print("statistical summary for numerical features:")
data.describe()

statistical summary for numerical features:


,unique_id,"On a scale of 1–10, how intense is the emotion conveyed by the artwork?",How many prominent colours do you notice in this painting?,How many objects caught your eye in the painting?
count,1686.000000,1621.000000,1612.000000,1612.000000
mean,281.500000,6.321900,3.878412,3.977047
std,162.283303,2.195637,3.495867,3.824206
min,1.000000,0.000000,0.000000,0.000000
25%,141.000000,5.000000,3.000000,2.000000
50%,281.500000,7.000000,3.000000,3.000000
75%,422.000000,8.000000,5.000000,5.000000
max,562.000000,10.000000,100.000000,100.000000


We have 14 Questions:

$\bullet$ **Q1**: On a scale of 1–10, how intense is the emotion conveyed by the artwork? <br>
$\bullet$ **Q2**: Describe how this painting makes you feel.<br>
$\bullet$ **Q3**: This art piece makes me feel sombre.<br>
$\bullet$ **Q4**: This art piece makes me feel content.<br>
$\bullet$ **Q5**: This art piece makes me feel calm.<br>
$\bullet$ **Q6**: This art piece makes me feel uneasy.<br>
$\bullet$ **Q7**: How many prominent colors do you notice in this painting?<br>
$\bullet$ **Q8**: How many objects caught your eye in the painting?<br>
$\bullet$ **Q9**: How much (in Canadian dollars) would you be willing to pay for this painting?<br>
$\bullet$ **Q10**: If you could purchase this painting, which room would you put that painting in?<br>
$\bullet$ **Q11**: If you could view this art in person, who would you want to view it with?<br>
$\bullet$ **Q12**: What season does this art piece remind you of?<br>
$\bullet$ **Q13**: If this painting was a food, what would be?<br>
$\bullet$ **Q14**: Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.



In [9]:
# Changing the name of the columns from the their respective questions to their question number:
data = data.rename(columns={
    "On a scale of 1–10, how intense is the emotion conveyed by the artwork?": "Q1",
    "Describe how this painting makes you feel.": "Q2",
    "This art piece makes me feel sombre.": "Q3",
    "This art piece makes me feel content.": "Q4",
    "This art piece makes me feel calm.": "Q5",
    "This art piece makes me feel uneasy.": "Q6",
    "How many prominent colours do you notice in this painting?": "Q7",
    "How many objects caught your eye in the painting?": "Q8",
    "How much (in Canadian dollars) would you be willing to pay for this painting?": "Q9",
    "If you could purchase this painting, which room would you put that painting in?": "Q10",
    "If you could view this art in person, who would you want to view it with?": "Q11", 
    "What season does this art piece remind you of?": "Q12",
    "If this painting was a food, what would be?": "Q13",
    "Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.": "Q14"
})

data.head()

,unique_id,Painting,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14
0,1,The Persistence of Memory,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,The Persistence of Memory,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,4.0,0,Bathroom,By yourself,Fall,Fries,A country song that contrasts nostalgia for th...
2,3,The Persistence of Memory,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,3.0,$5,"Bathroom,Dining room","Coworkers/Classmates,By yourself",Fall,Sardines,A melancholy instrumental with a monotone voic...
3,4,The Persistence of Memory,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,7.0,a,"Bedroom,Bathroom",Coworkers/Classmates,Winter,a,q
4,5,The Persistence of Memory,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,6.0,300 dollars.,Living room,Friends,"Spring,Summer",Churros.,Radiohead's album in rainbows.


We want to make this data as Numeric as possible, right now we only have 3 questions that is numeric. We can see that Q3, Q4, Q5, Q6, Q10, Q11, Q12 have responses that are categorical, so we can create indicator features for them:

**Making Q3, Q4, Q5, Q6 Numeric**

Technically, these questions are already numeric, they just have their numeric values paired with text we don't need, so we can just remove them for these questions:

In [10]:
# Creating new columns that just contain the numerical values:
data["Q3"] = (
    data["Q3"]
    .str.split(" - ")
    .str[0]
    .astype(float)
)

data["Q4"] = (
    data["Q4"]
    .str.split(" - ")
    .str[0]
    .astype(float)
)

data["Q5"] = (
    data["Q5"]
    .str.split(" - ")
    .str[0]
    .astype(float)
)

data["Q6"] = (
    data["Q6"]
    .str.split(" - ")
    .str[0]
    .astype(float)
)

**Making Q9 Numeric**

It seems that Q9 is basically a numeric question, but is made to be text, we are not interested in the text, so we can remove them:

In [11]:
# Converting responses to numeric only.
data["Q9"] = data["Q9"].str.extract(r'(\d+\.?\d*)').astype(float)

**Making Q10, Q11, Q12 Numeric**

These are questions that have categorical responses, so we can create indicator features for them.

In [12]:
# Creating indicator features for Q: If you could purchase this painting, which room would you put that painting in?
data["10a"] = data["Q10"].apply(lambda x: 1 if "Bedroom" in str(x) else 0)
data["10b"] = data["Q10"].apply(lambda x: 1 if "Bathroom" in str(x) else 0)
data["10c"] = data["Q10"].apply(lambda x: 1 if "Office" in str(x) else 0)
data["10d"] = data["Q10"].apply(lambda x: 1 if "Living room" in str(x) else 0)
data["10e"] = data["Q10"].apply(lambda x: 1 if "Dining room" in str(x) else 0)

# Creating indicator features for Q: If you could view this art in person, who would you want to view it with?
data["11a"] = data["Q11"].apply(lambda x: 1 if "Friends" in str(x) else 0)
data["11b"] = data["Q11"].apply(lambda x: 1 if "Family members" in str(x) else 0)
data["11c"] = data["Q11"].apply(lambda x: 1 if "Coworkers/Classmates" in str(x) else 0)
data["11d"] = data["Q11"].apply(lambda x: 1 if "Strangers" in str(x) else 0)
data["11e"] = data["Q11"].apply(lambda x: 1 if "By yourself" in str(x) else 0)

# Creating indicator features for Q: What season does this art piece remind you of?
data["12a"] = data["Q12"].apply(lambda x: 1 if "Spring" in str(x) else 0)
data["12b"] = data["Q12"].apply(lambda x: 1 if "Summer" in str(x) else 0)
data["12c"] = data["Q12"].apply(lambda x: 1 if "Fall" in str(x) else 0)
data["12d"] = data["Q12"].apply(lambda x: 1 if "Winter" in str(x) else 0)

# We can drop the original columns:
data.drop("Q10", axis=1, inplace=True)
data.drop("Q11", axis=1, inplace=True)
data.drop("Q12", axis=1, inplace=True)

# We dont want any entries where the respondent did not answer the question:
cols_10 = ["10a", "10b", "10c", "10d", "10e"]
cols_11 = ["11a", "11b", "11c", "11d", "11e"]
cols_12 = ["12a", "12b", "12c", "12d"]

data = data[data[cols_10].sum(axis=1) > 0]
data = data[data[cols_11].sum(axis=1) > 0]
data = data[data[cols_12].sum(axis=1) > 0]

Notice that we have some NaN entries. We should remove these entries:

In [13]:
data = data.dropna()

Making **Q13** Numeric

Q13 is different, it is completely text based, so a bag of words method might be the approach here. But we don't want a direct bag-of-words approach here, because we dont count every single word we see in a response, we only care about the foods, and categorizing them so we can create indicator features of the more popular foods that show up, then
see if a response has those foods or not. We can start by seeing which foods we have:

In [14]:
# Start by seeing how many responses of food we got.
food_counts = data["Q13"].value_counts().to_dict()
print(food_counts)

{'Salad': 43, 'salad': 25, 'Ice cream': 24, 'Pizza': 21, 'ice cream': 19, 'pizza': 15, 'Soup': 14, 'bread': 11, 'cake': 10, 'blueberry': 9, 'Cheese': 8, 'watermelon': 8, 'Pasta': 8, 'soup': 8, 'Steak': 8, 'Spaghetti': 8, 'Bread': 7, 'Sushi': 7, 'spaghetti': 6, 'Cake': 6, 'Sandwich': 6, 'Blueberry': 6, 'pasta': 6, 'Salad.': 6, 'Blueberry pie': 5, 'Strawberry': 5, 'apple': 5, 'Blueberries': 5, 'Apple pie': 5, 'Lemon': 5, 'noodles': 5, 'blueberry cheesecake': 5, 'ramen': 5, 'sushi': 5, 'blueberry pie': 5, 'Coffee': 5, 'A salad': 5, 'Cheesecake': 5, 'Ice Cream': 4, 'Burger': 4, 'hot chocolate': 4, 'stale bread': 4, 'banana': 4, 'Chicken noodle soup': 4, 'cheese': 4, 'shawarma': 3, 'A salad.': 3, 'cheesecake': 3, 'Poutine': 3, 'apple pie': 3, 'tiramisu': 3, 'Oatmeal': 3, 'Matcha cake': 3, 'Pear': 3, 'Orange': 3, 'Porridge': 3, 'toast': 3, 'Chocolate': 3, 'Cold soup': 3, 'Rice': 3, 'Lasagna': 3, 'icecream': 3, 'octopus': 3, 'Watermelon': 3, 'Lettuce': 3, 'Caesar salad': 3, 'Pancake': 3, 'Ban

We can see that there are a lot of duplicates that did not get paired together. Theres a "Salad" and "salad" value, being treated
separately, we want those paired together.

In [15]:
# This will group values like "Ice cream" and "ice cream together", "Pizza" and "a pizza".
data["Q13"] = (
    data["Q13"]
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z\s]', '', regex=True)   # remove punctuation
    .str.replace(r'\b(a|an|the)\b', '', regex=True)  # remove articles
    .str.replace(r'\s+', ' ', regex=True)  # clean extra spaces
)

We will have to manually group some values as well, below we replace certain texts to match out value entries, so that they can get paired together.

In [16]:
def normalize(text):
    text = str(text).lower().strip()

    # spelling mistakes
    text = text.replace("popscicle", "popsicle")
    text = text.replace("spagetti", "spaghetti")
    text = text.replace("sandwhich", "sandwich")
    text = text.replace("doughnut", "donut")
    text = text.replace("bluberries", "blueberry")
    text = text.replace("macha", "matcha")
    text = text.replace("yoghurt", "yogurt")
    text = text.replace("cantelope", "cantaloupe") 
    text = text.replace("pomegrante", "pomegranate") 
    text = text.replace("fettucine", "fettuccine")

    # common variants
    text = text.replace("blue berry", "blueberry")
    text = text.replace("blueberries", "blueberry")
    text = text.replace("blackberries", "blackberry")
    text = text.replace("raspberries", "raspberry")
    text = text.replace("strawberries", "strawberry")
    text = text.replace("hamburger", "burger")
    text = text.replace("cheeseburger", "burger")
    text = text.replace("cheesecake", "cake")
    text = text.replace("cupcake", "cake")
    text = text.replace("macaron", "cake")
    text = text.replace("cream puff", "cake")
    text = text.replace("brownie", "cake")
    text = text.replace("flan", "cake")
    text = text.replace("panna cotta", "cake")
    text = text.replace("creme brulee", "cake")
    text = text.replace("swiss roll", "cake")
    text = text.replace("croquembouche", "cake") 
    text = text.replace("tiramisu", "cake") 
    text = text.replace("parfait", "cake")  
    text = text.replace("meringue", "cake") 
    text = text.replace("muffin", "cake") 
    text = text.replace("creme brule", "cake") 
    text = text.replace("oreo", "cookie")
    text = text.replace("omelette", "egg")
    text = text.replace("jolly rancher", "candy")
    text = text.replace("lollipop", "candy")
    text = text.replace("carrot", "vegetable")
    text = text.replace("asparagus", "vegetable")
    text = text.replace("cabbage", "vegetable")
    text = text.replace("green peas", "vegetable")
    text = text.replace("boiled leafy greens", "vegetable")
    text = text.replace("sauteed greens", "vegetable")
    text = text.replace("onion", "vegetable")
    text = text.replace("pickle", "vegetable")
    text = text.replace("waffle", "pancake")
    text = text.replace("bagel", "bread")
    text = text.replace("bruschetta", "bread")
    text = text.replace("alcohol", "wine") 
    text = text.replace("whisky", "wine")
    text = text.replace("kool aid", "juice")
    text = text.replace("lemonade", "juice")
    text = text.replace("olive", "fruit")
    text = text.replace("starfruit", "fruit")
    text = text.replace("grapefruit", "fruit")
    text = text.replace("dragonfruit", "fruit")
    text = text.replace("durian", "fruit")
    text = text.replace("coconut", "fruit")
    text = text.replace("olive", "fruit")
    text = text.replace("cantaloupe", "fruit")
    text = text.replace("pear", "fruit")
    text = text.replace("pomegranate", "fruit")
    text = text.replace("pineapple", "fruit")
    text = text.replace("cherry", "fruit")
    text = text.replace("cherries", "fruit")
    text = text.replace("wintermelon", "melon")
    text = text.replace("sardines", "fish")
    text = text.replace("salmon", "fish")
    text = text.replace("tuna", "fish")
    text = text.replace("steak", "meat")
    text = text.replace("lamb", "meat")
    text = text.replace("mutton", "meat")
    text = text.replace("sausage", "meat")
    text = text.replace("prime rib", "meat")
    text = text.replace("filet mignon", "meat")
    text = text.replace("meatloaf", "meat")
    text = text.replace("cold", "iced")
    text = text.replace("cool", "iced")
    text = text.replace("snow", "iced")
    text = text.replace("freezie", "iced")
    text = text.replace("raisin", "grape")
    text = text.replace("french fries", "fry")
    text = text.replace("fries", "fry")
    text = text.replace("icecream", "ice cream")
    text = text.replace("gelato", "ice cream")
    text = text.replace("sorbet", "ice cream")
    text = text.replace("popsicle", "ice cream")
    text = text.replace("sundae", "ice cream")
    text = text.replace("latte", "coffee")
    text = text.replace("hot cocoa", "coffee")
    text = text.replace("teacoffee", "coffee")
    text = text.replace("chowder", "soup") 
    text = text.replace("fettuccine", "pasta")

    # General fixes
    text = text.replace("apples", "apple")
    text = text.replace("asparaguses", "asparagus")
    text = text.replace("avocados", "avocado")
    text = text.replace("bananas", "banana")
    text = text.replace("beans", "bean")
    text = text.replace("beefs", "beef")
    text = text.replace("breads", "bread")
    text = text.replace("broccolis", "broccoli")
    text = text.replace("burgers", "burger")
    text = text.replace("butters", "butter")
    text = text.replace("blueberries", "blueberry")
    text = text.replace("blackberries", "blackberry")

    text = text.replace("candys", "candy")
    text = text.replace("candies", "candy")
    text = text.replace("cakes", "cake")
    text = text.replace("caramels", "caramel")
    text = text.replace("carrots", "carrot")
    text = text.replace("caviars", "caviar")
    text = text.replace("cereals", "cereal")
    text = text.replace("cheeses", "cheese")
    text = text.replace("chickens", "chicken")
    text = text.replace("chocolates", "chocolate")
    text = text.replace("coffees", "coffee")
    text = text.replace("cookies", "cookie")
    text = text.replace("corns", "corn")
    text = text.replace("crabs", "crab")
    text = text.replace("croissants", "croissant")
    text = text.replace("cucumbers", "cucumber")
    text = text.replace("curries", "curry")

    text = text.replace("donuts", "donut")
    text = text.replace("ducks", "duck")
    text = text.replace("durians", "durian")

    text = text.replace("eggs", "egg")

    text = text.replace("fishs", "fish")
    text = text.replace("fishes", "fish")
    text = text.replace("flans", "flan")
    text = text.replace("fries", "fry")
    text = text.replace("fruits", "fruit")

    text = text.replace("garlics", "garlic")
    text = text.replace("gingers", "ginger")
    text = text.replace("geese", "goose")
    text = text.replace("grapes", "grape")
    text = text.replace("grapefruits", "grapefruit")
    text = text.replace("guacamoles", "guacamole")

    text = text.replace("hams", "ham")
    text = text.replace("honeys", "honey")
    text = text.replace("hummuses", "hummus")

    text = text.replace("ice creams", "ice cream")
    text = text.replace("iceds", "iced")

    text = text.replace("jellies", "jelly")
    text = text.replace("juices", "juice")

    text = text.replace("kimchis", "kimchi")
    text = text.replace("kiwis", "kiwi")

    text = text.replace("lambs", "lamb")
    text = text.replace("lasagnas", "lasagna")
    text = text.replace("lemons", "lemon")
    text = text.replace("lentils", "lentil")
    text = text.replace("lettuces", "lettuce")
    text = text.replace("lobsters", "lobster")

    text = text.replace("mangos", "mango")
    text = text.replace("meats", "meat")
    text = text.replace("matchas", "matcha")
    text = text.replace("milks", "milk")
    text = text.replace("mints", "mint")
    text = text.replace("mochis", "mochi")
    text = text.replace("muffins", "muffin")
    text = text.replace("mushrooms", "mushroom")

    text = text.replace("naans", "naan")
    text = text.replace("noodles", "noodle")

    text = text.replace("oatmeals", "oatmeal")
    text = text.replace("octopuses", "octopus")
    text = text.replace("olives", "olive")
    text = text.replace("omelettes", "omelette")
    text = text.replace("onions", "onion")
    text = text.replace("oranges", "orange")

    text = text.replace("pancakes", "pancake")
    text = text.replace("pastas", "pasta")
    text = text.replace("pastries", "pastry")
    text = text.replace("peaches", "peach")
    text = text.replace("pears", "pear")
    text = text.replace("peppers", "pepper")
    text = text.replace("pickles", "pickle")
    text = text.replace("pies", "pie")
    text = text.replace("pizzas", "pizza")
    text = text.replace("plums", "plum")
    text = text.replace("popcorns", "popcorn")
    text = text.replace("porks", "pork")
    text = text.replace("potatoes", "potato")
    text = text.replace("poutines", "poutine")
    text = text.replace("pretzels", "pretzel")
    text = text.replace("puddings", "pudding")
    text = text.replace("pumpkins", "pumpkin")

    text = text.replace("raspberries", "raspberry")
    text = text.replace("rices", "rice")

    text = text.replace("salads", "salad")
    text = text.replace("salmons", "salmon")
    text = text.replace("sandwiches", "sandwich")
    text = text.replace("sausages", "sausage")
    text = text.replace("seaweeds", "seaweed")
    text = text.replace("shrimps", "shrimp")
    text = text.replace("smoothies", "smoothie")
    text = text.replace("sodas", "soda")
    text = text.replace("soups", "soup")
    text = text.replace("spaghettis", "spaghetti")
    text = text.replace("spinaches", "spinach")
    text = text.replace("squids", "squid")
    text = text.replace("strawberries", "strawberry")
    text = text.replace("sugars", "sugar")
    text = text.replace("sushis", "sushi")

    text = text.replace("teas", "tea")
    text = text.replace("tacos", "taco")
    text = text.replace("toasts", "toast")
    text = text.replace("tomatoes", "tomato")
    text = text.replace("tunas", "tuna")
    text = text.replace("turkeys", "turkey")

    text = text.replace("vegetables", "vegetable")

    text = text.replace("waters", "water")
    text = text.replace("watermelons", "watermelon")
    text = text.replace("wines", "wine")

    text = text.replace("yogurts", "yogurt")

    return text

Now we will make a list of all the unique foods we saw in the responses, and go through each response and see if it contains a word we see in our list. For eg: "This painting reminds me of pizza", should be paired with "pizza". We will create a list of most popular foods, then go through each response and see if these words show up in the response. We then change the response to be a tuple of popular foods that were mentioned. If the response doesn't contain a popular food listed, the response is just sent back.

In [ ]:
# Now we will go through all the responses, and see if they contain a word that is present in our more popular responses:
top_foods = ['apple', 'asparagus','avocado',
             'banana','bean','beef','bread','broccoli','burger', 'butter', 'blueberry', 'blackberry', 
             'candy', 'cake','caramel','carrot','caviar','cereal','cheese','chicken','chocolate', 'coffee', 'cookie', 'corn', 'crab','croissant','cucumber', 'curry', 
             'donut','duck', 
             'egg', 
             'fish','fry','fruit', 
             'garlic','ginger','goose','grape','grapefruit','guacamole', 
             'ham','honey','hummus', 
             'ice cream', 'iced', 
             'jelly','juice', 
             'kimchi','kiwi', 
             'lamb','lasagna','lemon','lentil','lettuce', 'lobster', 
             'mango','meat', 'matcha','milk','mint','mochi','muffin','mushroom', 
             'naan','noodle', 
             'oatmeal','octopus','olive','omelette','onion','orange', 
             'pancake', 'pasta','pastry','peach','pear','pepper','pickle','pie','pizza','plum', 'popcorn','pork','potato','poutine','pretzel','pudding','pumpkin', 
             'raspberry', 'rice', 
             'salad', 'salmon','sandwich','sausage','seaweed','shrimp','smoothie', 'soda','soup','spaghetti','spinach','squid','strawberry','sugar', 'sushi', 
             'tea', 'taco', 'toast','tomato','tuna','turkey', 
             'vegetable', 
             'water','watermelon', 'wine', 
             'yogurt']

# Search through all the responses, see if any popular ones are present:

def map_food(text):
    text = normalize(text)
    matches = []

    for food in top_foods:
        if re.search(r'\b' + re.escape(food) + r'\b', text):
            matches.append(food)

    return tuple(matches) if matches else text

data["Q13"] = data["Q13"].apply(map_food)

In case, its not clear what map_food does:

In [18]:
print(map_food('chicken salad'))

('chicken', 'salad')


In [19]:
food_counts = data["Q13"].value_counts().to_dict()
print(food_counts)

{('salad',): 130, ('ice cream',): 89, ('cake',): 73, ('pizza',): 53, ('bread',): 51, ('soup',): 50, ('cheese',): 40, ('blueberry',): 35, ('fruit',): 32, ('spaghetti',): 29, ('meat',): 28, ('pasta',): 26, ('chocolate',): 24, ('sandwich',): 21, ('strawberry',): 19, ('noodle',): 19, ('apple',): 19, ('burger',): 16, ('blueberry', 'pie'): 14, ('tea',): 13, ('watermelon',): 13, ('fish',): 13, ('blueberry', 'cake'): 13, ('egg',): 12, ('coffee',): 12, ('vegetable',): 12, ('sushi',): 12, ('beef',): 11, ('rice',): 11, ('cake', 'chocolate'): 11, ('apple', 'pie'): 11, ('potato',): 9, ('banana',): 9, ('chicken',): 9, ('pie',): 9, ('iced',): 9, ('candy',): 9, ('pancake',): 8, ('fruit', 'salad'): 8, ('water',): 8, ('cake', 'matcha'): 8, ('cookie',): 8, 'ramen': 7, ('chicken', 'noodle', 'soup'): 7, ('toast',): 7, ('lasagna',): 7, ('iced', 'soup'): 7, ('bean',): 7, ('grape',): 7, ('blueberry', 'ice cream'): 7, ('wine',): 6, ('orange',): 6, ('croissant',): 6, ('juice',): 6, ('lettuce',): 6, ('taco',): 6

The column now tracks which combination of foods appear the most, we need to now shift through this data, and find: <br>
$\bullet$ Which individual foods are most common (i.e which foods show up in responses at least 5 times) <br>
$\bullet$ Then create indicator features for the foods that appear at least 5 times <br>
$\bullet$ Then drop any entries that didn't list a food that was popular enough to become an indicator feature. <br>

In [ ]:
mapped = data["Q13"].apply(map_food)

food_counts = Counter()
for entry in mapped:
    if isinstance(entry, tuple):
        food_counts.update(entry)
    else:
        food_counts.update([entry])

popular_foods = {food for food, count in food_counts.items() if count >= 5}

for food in popular_foods:
    col_name = f"food_{food.replace(' ', '_')}"
    
    data[col_name] = mapped.apply(lambda x: int(food in x) if isinstance(x, tuple) else int(food == x))

indicator_cols = [col for col in data.columns if col.startswith("food_")]

data = data[(data[indicator_cols].sum(axis=1) > 0)]

In [21]:
# We can drop the original column:
data.drop("Q13", axis=1, inplace=True)

**Making the Output Numeric**

We have three paintings that a new input can be classified as: "The Persistence of Memory", "The Water Lily Pond", "The Starry Night". We can map these as 1,2,3:

In [26]:
# Create new mappings
painting_mapping = {"The Persistence of Memory": 1, "The Water Lily Pond": 2, "The Starry Night": 3}

# Map paintings to numerical values.
data["Painting"] = data["Painting"].map(painting_mapping)

In [27]:
data.head()

,unique_id,Painting,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,...,food_pancake,food_banana,food_lemon,food_wine,food_blueberry,food_fish,food_sandwich,food_sushi,food_pasta,food_egg
1,2,1,5.0,"The clocks are burnt on a hot desert, it embod...",4.0,3.0,2.0,1.0,2.0,4.0,...,0,0,0,0,0,0,0,0,0,0
2,3,1,7.0,This painting makes me feel dread. The clock r...,4.0,1.0,1.0,4.0,4.0,3.0,...,0,0,0,0,0,1,0,0,0,0
5,6,1,7.0,This painting makes me feel a bit confused and...,4.0,2.0,1.0,5.0,5.0,6.0,...,0,0,0,0,0,0,0,0,0,0
6,7,1,8.0,It has a sorrow feeling and clock is melting f...,4.0,2.0,2.0,4.0,3.0,3.0,...,0,0,0,0,0,0,0,0,0,0
7,8,1,4.0,"The painting makes me feel somewhat sad, think...",4.0,2.0,2.0,4.0,7.0,8.0,...,0,0,0,0,0,1,0,0,0,0


In [28]:
data.shape

(1369, 92)

In [29]:
print(f'Number of columns: {len(data.columns)}')
for column_name in data.columns:
    print(column_name)

Number of columns: 92
unique_id
Painting
Q1
Q2
Q3
Q4
Q5
Q6
Q7
Q8
Q9
Q14
10a
10b
10c
10d
10e
11a
11b
11c
11d
11e
12a
12b
12c
12d
food_lettuce
food_tomato
food_fry
food_pudding
food_broccoli
food_seaweed
food_ice_cream
food_orange
food_vegetable
food_butter
food_pizza
food_burger
food_chicken
food_yogurt
food_spaghetti
food_ramen
food_iced
food_turkey
food_smoothie
food_taco
food_grape
food_spinach
food_matcha
food_croissant
food_cheese
food_meat
food_rice
food_cookie
food_cake
food_toast
food_oatmeal
food_bean
food_watermelon
food_bread
food_apple
food_tea
food_sugar
food_lasagna
food_salad
food_candy
food_beef
food_noodle
food_chocolate
food_honey
food_milk
food_strawberry
food_soup
food_water
food_juice
food_cucumber
food_coffee
food_potato
food_fruit
food_pie
food_cereal
food_mango
food_pancake
food_banana
food_lemon
food_wine
food_blueberry
food_fish
food_sandwich
food_sushi
food_pasta
food_egg
